In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive/dataforsign/data"))

['hello', 'one', 'peace', 'thumbsup', 'yes']


In [3]:
import cv2
import os
import numpy as np

video_path = "/content/drive/MyDrive/dataforsign/data"
output_path = "/content/drive/MyDrive/dataforsign/static_dataset"

IMG_SIZE = 96
FRAMES_PER_VIDEO = 30

os.makedirs(output_path, exist_ok=True)

for gesture in os.listdir(video_path):
    gesture_video_path = os.path.join(video_path, gesture)
    gesture_output_path = os.path.join(output_path, gesture)
    os.makedirs(gesture_output_path, exist_ok=True)

    img_count = 0

    for vid in os.listdir(gesture_video_path):
        cap = cv2.VideoCapture(os.path.join(gesture_video_path, vid))

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        start_frame = int(0.2 * total_frames)
        end_frame = int(0.8 * total_frames)

        frame_indices = np.linspace(
            start_frame,
            end_frame - 1,
            FRAMES_PER_VIDEO,
            dtype=int
        )

        current_frame = 0
        selected_idx = 0

        while cap.isOpened() and selected_idx < FRAMES_PER_VIDEO:
            ret, frame = cap.read()
            if not ret:
                break

            if current_frame == frame_indices[selected_idx]:
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                resized = cv2.resize(gray, (IMG_SIZE, IMG_SIZE))

                save_path = os.path.join(
                    gesture_output_path,
                    f"{gesture}_{img_count}.jpg"
                )
                cv2.imwrite(save_path, resized)

                img_count += 1
                selected_idx += 1

            current_frame += 1

        cap.release()

print("Middle-region extraction complete")

Middle-region extraction complete


In [4]:
for g in os.listdir(output_path):
    print(g, len(os.listdir(os.path.join(output_path, g))))

hello 300
one 300
peace 300
thumbsup 300
yes 300


In [5]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models

data_path = "/content/drive/MyDrive/dataforsign/static_dataset"
IMG_SIZE = 96

CLASSES = sorted([
    d for d in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path, d))
])
print("Classes:", CLASSES)

X = []
y = []

for idx, cls in enumerate(CLASSES):
    cls_path = os.path.join(data_path, cls)
    for img in os.listdir(cls_path):
        if not img.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img_path = os.path.join(cls_path, img)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            print("Skipping unreadable image:", img_path)
            continue

        X.append(image)
        y.append(idx)

X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
X = X / 255.0
y = np.array(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Classes: ['hello', 'one', 'peace', 'thumbsup', 'yes']
Train shape: (1200, 96, 96, 1)
Val shape: (300, 96, 96, 1)


In [6]:
model = models.Sequential([

    layers.Conv2D(32, (3,3), activation='relu', input_shape=(96,96,1)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 20, 20, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12800)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,638,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,731,845 (6.61 MB)

 Trainable params: 1,731,845 (6.61 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
print(len(CLASSES))

5


In [8]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    validation_data=(X_val, y_val)
)

Epoch 1/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 31s 669ms/step - accuracy: 0.5408 - loss: 1.1402 - val_accuracy: 0.9267 - val_loss: 0.2389
Epoch 2/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 40s 681ms/step - accuracy: 0.9442 - loss: 0.1688 - val_accuracy: 0.9967 - val_loss: 0.0348
Epoch 3/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 43s 714ms/step - accuracy: 0.9850 - loss: 0.0503 - val_accuracy: 0.9967 - val_loss: 0.0134
Epoch 4/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 38s 638ms/step - accuracy: 0.9875 - loss: 0.0438 - val_accuracy: 1.0000 - val_loss: 0.0032
Epoch 5/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 24s 643ms/step - accuracy: 0.9950 - loss: 0.0177 - val_accuracy: 1.0000 - val_loss: 7.5041e-04
Epoch 6/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 39s 596ms/step - accuracy: 0.9967 - loss: 0.0099 - val_accuracy: 1.0000 - val_loss: 7.1494e-04
Epoch 7/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 25s 672ms/step - accuracy: 0.9992 - loss: 0.0040 - val_accuracy: 1.0000 - val_loss: 1.4326e-04
Epoch 8/15
38/38 ━━━━━━━━━━━━━━━━━━━━ 40s 658ms/step - accuracy: 0.9992 - loss: 0.006

In [9]:
model.evaluate(X_train, y_train)
model.evaluate(X_val, y_val)

38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 137ms/step - accuracy: 1.0000 - loss: 0.0018
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 215ms/step - accuracy: 0.9967 - loss: 0.0076


[0.007614798378199339, 0.996666669845581]

In [10]:
model.save("/content/drive/MyDrive/dataforsign/sign_model_5class.h5")